# Model Stability Audit For Logistic Regression And CatBoost: Repeated Customer-Grouped Splits

This notebook repeats the model stability audit with two further classifiers, Logistic Regression and CatBoost, next to the Random Forest of the earlier audit. The six experiment cells (three feature sets, two releases) are evaluated on the same repeated customer-grouped splits, so that the comparison between the source release and the anonymised release can be examined for more than one type of model.

The earlier audit used only the default Random Forest of notebook 03, so a conclusion about the effect of anonymisation could depend on that single model family. Here the same procedure is applied without any tuning to a linear model and to a gradient boosting model, and the paired differences between the releases are reported for each of them.

## Design

- **Splits.** For each of the 50 seeds (1 to 51, without 42, the seed of the modelling notebooks) the table is partitioned as in the modelling notebooks: two successive grouped shuffle splits by `Customer_ID` (70% train, then the remainder halved into validation and test), using the source customer labels. The same partitions are applied to both releases through `Property_ID`. All three models use the same 50 seeds. The number of splits used by CatBoost is set by `N_CATBOOST_SPLITS` in the first code cell, which can be lowered to shorten the run, and every paired comparison of a model uses the splits of that model.
- **Models.** The Random Forest is the one of notebook 03 (500 trees, balanced subsample class weights, seed 42, all other settings at their defaults). Logistic Regression is the one of notebook 03 (at most 5,000 iterations, seed 42, all other settings at the library defaults, which include L2 regularisation with C equal to 1.0). CatBoost uses the multi-class loss, balanced class weights and seed 42, with all hyperparameters at the library defaults. There is no search, no tuning and no selection for any of the three models, and the validation partition is not used.
- **Preprocessing.** The same for all models and the same as in the modelling notebooks: median imputation and scaling for numeric columns, mode imputation and one-hot encoding for the others, fitted on the training partition only.
- **Reference.** A class-prior baseline ranks the classes by their frequency in the training partition.
- **Paired comparisons.** Differences are computed within each split and for each model: anonymised release minus source release for each feature set, the full feature set minus the no-AI feature set (with AI minus without AI), and the pure-record feature set minus the full feature set within each release.
- **What is reported.** Mean, standard deviation and the 2.5th and 97.5th percentiles over the splits. The values are descriptive. The splits are drawn from the same 431 states and overlap, so the spread is not the sampling variability of a new set of customers, and no significance statement is made. The test partitions differ in size from split to split.
- **Comparability.** The CatBoost hyperparameters are the library defaults and not the values selected by the searches of notebook 03, so the CatBoost scores of this notebook are not comparable with the CatBoost rows of the modelling notebooks.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

DATA_DIR = Path("../../data/processed/hypothesis_testing")

SEEDS = [seed for seed in range(1, 52) if seed != 42]
CHECK_SEED = 42

# Number of splits used by CatBoost, which is the slowest of the three models. Lower it to shorten the run.
N_CATBOOST_SPLITS = 50

## Data

Both modelling tables have the same rows in the same order. The check below confirms that the keys, timestamps and targets are identical in the two releases, so the same row indices can be used for both.

In [2]:
source = pd.read_csv(DATA_DIR / "source_dataset" / "model_state.csv", parse_dates=["Prediction_Time"])
regions = pd.read_csv(DATA_DIR / "regions_anonymised" / "model_state.csv", parse_dates=["Prediction_Time"])

for column in ["Property_ID", "Prediction_Time", "Next_Category"]:
    assert (source[column].values == regions[column].values).all(), column

RELEASES = {"source": source, "regions": regions}

print("DATA LOADED")
print("-" * 60)
print(f"Rows:       {len(source):,}")
print(f"Properties: {source['Property_ID'].nunique():,}")
print(f"Customers:  {source['Customer_ID'].nunique():,}")

DATA LOADED
------------------------------------------------------------
Rows:       431
Properties: 234
Customers:  198


## Feature Sets And Models

The three feature sets are the ones used in the modelling notebooks: all 55 predictors, the 42 predictors without the `ai_` columns, and the 12 documented-record predictors. The three models are defined below with fixed settings. Logistic Regression is fitted with the settings of notebook 03, which raise the maximum number of iterations to 5,000 and leave every other setting at the scikit-learn default, namely the L-BFGS solver with L2 regularisation at an inverse strength C of 1.0. CatBoost is fitted with the multi-class loss, balanced class weights and a fixed seed of 42 as in the searches of notebook 03, but with all remaining hyperparameters at the CatBoost library defaults (1,000 iterations, a tree depth of 6, an automatically chosen learning rate and an L2 leaf regularisation of 3.0), so that no search or selection is involved.

In [3]:
IDENTIFIERS = ["Property_ID", "Customer_ID", "Prediction_Time", "Next_Category_ID", "Next_Category"]

ALL_FEATURES = [column for column in source.columns if column not in IDENTIFIERS]
NO_AI_FEATURES = [column for column in ALL_FEATURES if not column.startswith("ai_")]
RECORD_FEATURES = [
    "documented_asset_count",
    "documented_category_count",
    "documented_value_total",
    "documented_replacement_cost_total",
    "mean_evidence_rating",
    "documented_count_CAT-001",
    "documented_count_CAT-002",
    "documented_count_CAT-003",
    "documented_count_CAT-004",
    "documented_count_CAT-005",
    "documented_count_CAT-006",
    "documented_count_CAT-007",
]

FEATURE_SETS = {
    "full": ALL_FEATURES,
    "no_ai": NO_AI_FEATURES,
    "pure_record": RECORD_FEATURES,
}

assert [len(columns) for columns in FEATURE_SETS.values()] == [55, 42, 12]

# The Random Forest of notebook 03: no search, no tuning.
FOREST_SETTINGS = dict(
    n_estimators=500,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

# The Logistic Regression of notebook 03: no search, no tuning, library defaults otherwise.
LOGISTIC_SETTINGS = dict(
    max_iter=5000,
    random_state=42,
)

# CatBoost with the library default hyperparameters: no search, no tuning.
CATBOOST_SETTINGS = dict(
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
    thread_count=-1,
)

MODELS = {
    "random_forest": (RandomForestClassifier, FOREST_SETTINGS),
    "logistic": (LogisticRegression, LOGISTIC_SETTINGS),
    "catboost": (CatBoostClassifier, CATBOOST_SETTINGS),
}

MODEL_LABELS = {
    "random_forest": "Random Forest",
    "logistic": "Logistic Regression",
    "catboost": "CatBoost",
}

MODEL_SEEDS = {
    "random_forest": SEEDS,
    "logistic": SEEDS,
    "catboost": SEEDS[:N_CATBOOST_SPLITS],
}

print("FEATURE SETS")
print("-" * 60)
for name, columns in FEATURE_SETS.items():
    print(f"{name:<12} {len(columns)} predictors")

print()
print("MODELS")
print("-" * 60)
for name, (_, settings) in MODELS.items():
    print(f"{MODEL_LABELS[name]:<20} {len(MODEL_SEEDS[name])} splits | {settings}")

FEATURE SETS
------------------------------------------------------------
full         55 predictors
no_ai        42 predictors
pure_record  12 predictors

MODELS
------------------------------------------------------------
Random Forest        50 splits | {'n_estimators': 500, 'class_weight': 'balanced_subsample', 'random_state': 42, 'n_jobs': -1}
Logistic Regression  50 splits | {'max_iter': 5000, 'random_state': 42}
CatBoost             50 splits | {'loss_function': 'MultiClass', 'auto_class_weights': 'Balanced', 'random_seed': 42, 'verbose': 0, 'allow_writing_files': False, 'thread_count': -1}


## Check Against The Saved Models

The feature sets and the Random Forest above are written out in this notebook. The cell below loads the six saved Top-1 models of the modelling notebooks and confirms that each one was trained on exactly these predictors and with exactly these forest settings, so that nothing in this notebook rests on an unchecked assumption. Logistic Regression and CatBoost have no saved models in the modelling notebooks, so they share the feature sets, preprocessing and splits defined here and are checked through the consistency of the pure-record results between the two releases further below.

In [4]:
import joblib

MODELS_DIR = Path("../../models/hypothesis_testing")

print("CHECK AGAINST THE SAVED TOP-1 MODELS")
print("-" * 80)

for release in RELEASES:
    for feature_set, columns in FEATURE_SETS.items():
        saved = joblib.load(MODELS_DIR / f"{release}_{feature_set}_top1.joblib")

        saved_columns = [
            column
            for name, _, group in saved.named_steps["preprocessor"].transformers_
            if name in ("numerical", "categorical")
            for column in group
        ]

        saved_forest = saved.named_steps["classifier"].get_params()

        assert set(saved_columns) == set(columns), (release, feature_set)
        assert len(saved_columns) == len(columns), (release, feature_set)

        for name, value in FOREST_SETTINGS.items():
            assert saved_forest[name] == value, (release, feature_set, name)

        print(
            f"{release:<8} {feature_set:<12} predictors {len(saved_columns):>2} match | "
            f"forest settings match ({saved_forest['n_estimators']} trees)"
        )

print()
print("The saved Top-1 models use exactly the predictors and settings defined above.")

CHECK AGAINST THE SAVED TOP-1 MODELS
--------------------------------------------------------------------------------
source   full         predictors 55 match | forest settings match (500 trees)
source   no_ai        predictors 42 match | forest settings match (500 trees)
source   pure_record  predictors 12 match | forest settings match (500 trees)
regions  full         predictors 55 match | forest settings match (500 trees)
regions  no_ai        predictors 42 match | forest settings match (500 trees)
regions  pure_record  predictors 12 match | forest settings match (500 trees)

The saved Top-1 models use exactly the predictors and settings defined above.


## Functions

The preprocessing is the one of the modelling notebooks (median imputation and scaling for numeric columns, mode imputation and one-hot encoding for the others) and is fitted on the training partition only. The classifier is chosen by name. Top-k accuracy is computed from the ranked class probabilities, and a true class that is absent from a training partition counts as a miss.

In [5]:
def build_pipeline(frame, columns, model_name="random_forest"):
    numeric = frame[columns].select_dtypes(include=["number"]).columns.tolist()
    categorical = [column for column in columns if column not in numeric]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
                numeric,
            ),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical,
            ),
        ]
    )

    classifier_class, settings = MODELS[model_name]

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier_class(**settings)),
    ])


def top_k_accuracy(model, X, y, k):
    probabilities = model.predict_proba(X)
    classes = np.asarray(model.named_steps["classifier"].classes_)
    ranked = classes[np.argsort(-probabilities, axis=1, kind="stable")[:, :k]]
    return float(np.mean([truth in row for truth, row in zip(y, ranked)]))


def customer_grouped_split(seed):
    groups = source["Customer_ID"]
    target = source["Next_Category"]

    train_idx, temp_idx = next(
        GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=seed).split(
            source, target, groups=groups
        )
    )

    val_rel, test_rel = next(
        GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=seed).split(
            source.iloc[temp_idx], target.iloc[temp_idx], groups=groups.iloc[temp_idx]
        )
    )

    return train_idx, temp_idx[val_rel], temp_idx[test_rel]


def class_prior_baseline(y_train, y_test):
    counts = y_train.value_counts()
    order = sorted(counts.index, key=lambda label: (-counts[label], label))

    predictions = np.repeat(order[0], len(y_test))

    return dict(
        top1_accuracy=accuracy_score(y_test, predictions),
        macro_f1=f1_score(y_test, predictions, average="macro", zero_division=0),
        weighted_f1=f1_score(y_test, predictions, average="weighted", zero_division=0),
        top2_accuracy=float(np.mean([truth in order[:2] for truth in y_test])),
        top3_accuracy=float(np.mean([truth in order[:3] for truth in y_test])),
    )


def evaluate(release, feature_set, train_idx, test_idx, model_name="random_forest"):
    frame = RELEASES[release]
    columns = FEATURE_SETS[feature_set]
    X_train, X_test = frame[columns].iloc[train_idx], frame[columns].iloc[test_idx]
    y_train, y_test = frame["Next_Category"].iloc[train_idx], frame["Next_Category"].iloc[test_idx]

    model = build_pipeline(frame, columns, model_name).fit(X_train, y_train)
    predictions = np.asarray(model.predict(X_test)).ravel()

    return dict(
        top1_accuracy=accuracy_score(y_test, predictions),
        macro_f1=f1_score(y_test, predictions, average="macro", zero_division=0),
        weighted_f1=f1_score(y_test, predictions, average="weighted", zero_division=0),
        top2_accuracy=top_k_accuracy(model, X_test, y_test.values, 2),
        top3_accuracy=top_k_accuracy(model, X_test, y_test.values, 3),
    )

## Reproduction Check

Before the repetitions, the procedure is run for the Random Forest on the seed-42 split used in the modelling notebooks. The Top-1 metrics of every cell must equal the holdout results saved by the corresponding notebook, which confirms that the splits, preprocessing and Top-1 model are the same as in the modelling notebooks and in the earlier audit. Logistic Regression and CatBoost reuse exactly the same splits, preprocessing and evaluation functions.

In [6]:
FEATURE_FOLDERS = {
    "full": "01_full_feature_set",
    "no_ai": "02_no_ai_features",
    "pure_record": "03_pure_record",
}
RELEASE_FOLDERS = {"source": "source_dataset", "regions": "regions_anonymised"}

train_idx, val_idx, test_idx = customer_grouped_split(CHECK_SEED)

print("REPRODUCTION OF THE SEED-42 TOP-1 RESULTS (RANDOM FOREST)")
print("-" * 80)

for release, release_folder in RELEASE_FOLDERS.items():
    for feature_set, feature_folder in FEATURE_FOLDERS.items():
        saved = pd.read_csv(
            DATA_DIR / release_folder / feature_folder / "results_holdout_test.csv"
        ).iloc[0]

        result = evaluate(release, feature_set, train_idx, test_idx, "random_forest")

        differences = {
            "top1_accuracy": abs(result["top1_accuracy"] - saved["Top1_Accuracy"]),
            "macro_f1": abs(result["macro_f1"] - saved["Top1_Macro_F1"]),
            "weighted_f1": abs(result["weighted_f1"] - saved["Top1_Weighted_F1"]),
        }

        print(
            f"{release:<8} {feature_set:<12} "
            f"accuracy {result['top1_accuracy']:.4f} | Macro F1 {result['macro_f1']:.4f} | "
            f"max difference to saved {max(differences.values()):.2e}"
        )

        assert max(differences.values()) < 1e-9, (release, feature_set)

print()
print("All Top-1 metrics reproduce the saved holdout results.")

REPRODUCTION OF THE SEED-42 TOP-1 RESULTS (RANDOM FOREST)
--------------------------------------------------------------------------------
source   full         accuracy 0.4915 | Macro F1 0.3683 | max difference to saved 5.55e-17
source   no_ai        accuracy 0.5424 | Macro F1 0.4085 | max difference to saved 5.55e-17
source   pure_record  accuracy 0.4576 | Macro F1 0.3452 | max difference to saved 0.00e+00
regions  full         accuracy 0.4915 | Macro F1 0.3620 | max difference to saved 5.55e-17
regions  no_ai        accuracy 0.4915 | Macro F1 0.3380 | max difference to saved 5.55e-17
regions  pure_record  accuracy 0.4576 | Macro F1 0.3452 | max difference to saved 0.00e+00

All Top-1 metrics reproduce the saved holdout results.


## Repeated Splits

For each seed the models are refitted on the training partition and scored on the test partition, for both releases and all three feature sets. Nothing is tuned or selected on any split. The progress is printed every ten splits.

In [7]:
records = []
baselines = []

for number, seed in enumerate(SEEDS, start=1):
    train_idx, val_idx, test_idx = customer_grouped_split(seed)

    y_train = source["Next_Category"].iloc[train_idx]
    y_test = source["Next_Category"].iloc[test_idx]

    baselines.append(dict(seed=seed, **class_prior_baseline(y_train, y_test)))

    for model_name in MODELS:
        if seed not in MODEL_SEEDS[model_name]:
            continue

        for release in RELEASES:
            for feature_set in FEATURE_SETS:
                records.append(
                    dict(
                        seed=seed,
                        model=model_name,
                        release=release,
                        feature_set=feature_set,
                        test_rows=len(test_idx),
                        test_customers=int(source["Customer_ID"].iloc[test_idx].nunique()),
                        **evaluate(release, feature_set, train_idx, test_idx, model_name),
                    )
                )

    if number % 10 == 0:
        print(f"{number} of {len(SEEDS)} splits done")

per_split = pd.DataFrame(records)
baseline_per_split = pd.DataFrame(baselines)


print()
print("REPEATED SPLITS COMPLETE")
print("-" * 60)
print(f"Result rows:          {len(per_split)}")
for model_name in MODELS:
    group = per_split[per_split.model == model_name]
    print(
        f"{MODEL_LABELS[model_name]:<20} {group.seed.nunique():>2} splits | "
        f"test rows min {group['test_rows'].min()} | mean {group['test_rows'].mean():.1f} | max {group['test_rows'].max()}"
    )

10 of 50 splits done
20 of 50 splits done
30 of 50 splits done
40 of 50 splits done
50 of 50 splits done

REPEATED SPLITS COMPLETE
------------------------------------------------------------
Result rows:          900
Random Forest        50 splits | test rows min 52 | mean 66.9 | max 83
Logistic Regression  50 splits | test rows min 52 | mean 66.9 | max 83
CatBoost             50 splits | test rows min 52 | mean 66.9 | max 83


## Spread Of Each Cell

Mean, standard deviation and the 2.5th and 97.5th percentiles of every metric over the splits of each model. The class-prior baseline is the same for both releases and all models because it uses only the training labels, and it is computed over all 50 splits.

In [8]:
METRICS = {
    "top1_accuracy": "Top-1 accuracy",
    "macro_f1": "Macro F1",
    "weighted_f1": "Weighted F1",
    "top2_accuracy": "Top-2 accuracy",
    "top3_accuracy": "Top-3 accuracy",
}


def describe(values):
    return pd.Series(
        {
            "mean": values.mean(),
            "sd": values.std(ddof=1),
            "p2.5": values.quantile(0.025),
            "p97.5": values.quantile(0.975),
        }
    )


spread_rows = []

for (model_name, release, feature_set), group in per_split.groupby(["model", "release", "feature_set"]):
    for column, label in METRICS.items():
        spread_rows.append(
            dict(
                model=MODEL_LABELS[model_name],
                release=release,
                feature_set=feature_set,
                metric=label,
                **describe(group[column]),
            )
        )

for column, label in METRICS.items():
    spread_rows.append(
        dict(
            model="class-prior baseline",
            release="both",
            feature_set="-",
            metric=label,
            **describe(baseline_per_split[column]),
        )
    )

spread = pd.DataFrame(spread_rows)

print("SPREAD OVER THE SPLITS")
print("-" * 100)

for (model, release, feature_set), group in spread.groupby(["model", "release", "feature_set"], sort=False):
    print(f"\n{model} | {release} | {feature_set}")
    print(group.drop(columns=["model", "release", "feature_set"]).round(4).to_string(index=False))

SPREAD OVER THE SPLITS
----------------------------------------------------------------------------------------------------

CatBoost | regions | full
        metric   mean     sd   p2.5  p97.5
Top-1 accuracy 0.4641 0.0530 0.3723 0.5675
      Macro F1 0.3793 0.0697 0.2743 0.5394
   Weighted F1 0.4360 0.0521 0.3456 0.5391
Top-2 accuracy 0.7585 0.0393 0.6646 0.8203
Top-3 accuracy 0.8953 0.0266 0.8494 0.9452

CatBoost | regions | no_ai
        metric   mean     sd   p2.5  p97.5
Top-1 accuracy 0.4838 0.0454 0.4042 0.5643
      Macro F1 0.4015 0.0626 0.3073 0.5272
   Weighted F1 0.4571 0.0460 0.3872 0.5406
Top-2 accuracy 0.7711 0.0417 0.6947 0.8566
Top-3 accuracy 0.9012 0.0342 0.8311 0.9559

CatBoost | regions | pure_record
        metric   mean     sd   p2.5  p97.5
Top-1 accuracy 0.4018 0.0494 0.3157 0.4925
      Macro F1 0.3310 0.0593 0.2265 0.4241
   Weighted F1 0.3896 0.0472 0.3056 0.4816
Top-2 accuracy 0.6898 0.0461 0.6027 0.7755
Top-3 accuracy 0.8540 0.0329 0.7981 0.9128

CatBoost | s

## Paired Differences Between The Releases

Within each split and for each model, the anonymised release minus the source release for each feature set. A positive value means that the anonymised release scored higher. The last columns give the share of splits with a negative, zero and positive difference.

In [9]:
def paired_differences(a, b, comparison):
    rows = []
    for column, label in METRICS.items():
        difference = (a.set_index("seed")[column] - b.set_index("seed")[column]).dropna()
        rows.append(
            dict(
                comparison=comparison,
                metric=label,
                splits=len(difference),
                mean_difference=difference.mean(),
                sd=difference.std(ddof=1),
                **{"p2.5": difference.quantile(0.025), "p97.5": difference.quantile(0.975)},
                share_negative=float((difference < -1e-12).mean()),
                share_zero=float((difference.abs() <= 1e-12).mean()),
                share_positive=float((difference > 1e-12).mean()),
            )
        )
    return rows


def select(model_name, release, feature_set):
    return per_split[
        (per_split.model == model_name)
        & (per_split.release == release)
        & (per_split.feature_set == feature_set)
    ]


release_differences = []
for model_name in MODELS:
    for feature_set in FEATURE_SETS:
        release_differences += [
            dict(model=MODEL_LABELS[model_name], feature_set=feature_set, **row)
            for row in paired_differences(
                select(model_name, "regions", feature_set),
                select(model_name, "source", feature_set),
                "regions minus source",
            )
        ]

release_differences = pd.DataFrame(release_differences)

print("PAIRED DIFFERENCES: REGIONS MINUS SOURCE")
print("-" * 110)

for (model, feature_set), group in release_differences.groupby(["model", "feature_set"], sort=False):
    print(f"\n{model} | {feature_set}")
    print(group.drop(columns=["model", "feature_set", "comparison"]).round(4).to_string(index=False))

PAIRED DIFFERENCES: REGIONS MINUS SOURCE
--------------------------------------------------------------------------------------------------------------

Random Forest | full
        metric  splits  mean_difference     sd    p2.5  p97.5  share_negative  share_zero  share_positive
Top-1 accuracy      50           0.0006 0.0307 -0.0563 0.0543            0.40        0.12            0.48
      Macro F1      50           0.0044 0.0339 -0.0535 0.0794            0.44        0.00            0.56
   Weighted F1      50           0.0028 0.0338 -0.0524 0.0651            0.42        0.00            0.58
Top-2 accuracy      50          -0.0043 0.0362 -0.0689 0.0649            0.42        0.24            0.34
Top-3 accuracy      50          -0.0053 0.0289 -0.0555 0.0429            0.50        0.16            0.34

Random Forest | no_ai
        metric  splits  mean_difference     sd    p2.5  p97.5  share_negative  share_zero  share_positive
Top-1 accuracy      50           0.0093 0.0287 -0.0357 0.0552

## Paired Differences Between Feature Sets

Within each model, release and split: the full feature set minus the no-AI feature set (with AI minus without AI), and the pure-record feature set minus the full feature set.

In [10]:
feature_set_differences = []

for model_name in MODELS:
    for release in RELEASES:
        full = select(model_name, release, "full")
        feature_set_differences += [
            dict(model=MODEL_LABELS[model_name], release=release, **row)
            for row in paired_differences(full, select(model_name, release, "no_ai"), "with AI minus without AI (full minus no-AI)")
        ]
        feature_set_differences += [
            dict(model=MODEL_LABELS[model_name], release=release, **row)
            for row in paired_differences(select(model_name, release, "pure_record"), full, "pure record minus full")
        ]

feature_set_differences = pd.DataFrame(feature_set_differences)

print("PAIRED DIFFERENCES: FEATURE SETS WITHIN A RELEASE")
print("-" * 110)

for (model, release, comparison), group in feature_set_differences.groupby(["model", "release", "comparison"], sort=False):
    print(f"\n{model} | {release} | {comparison}")
    print(group.drop(columns=["model", "release", "comparison"]).round(4).to_string(index=False))

PAIRED DIFFERENCES: FEATURE SETS WITHIN A RELEASE
--------------------------------------------------------------------------------------------------------------

Random Forest | source | with AI minus without AI (full minus no-AI)
        metric  splits  mean_difference     sd    p2.5  p97.5  share_negative  share_zero  share_positive
Top-1 accuracy      50          -0.0162 0.0351 -0.0794 0.0469            0.58        0.20            0.22
      Macro F1      50          -0.0274 0.0518 -0.1613 0.0394            0.76        0.00            0.24
   Weighted F1      50          -0.0221 0.0360 -0.0884 0.0413            0.76        0.00            0.24
Top-2 accuracy      50          -0.0151 0.0386 -0.0987 0.0586            0.60        0.14            0.26
Top-3 accuracy      50          -0.0046 0.0299 -0.0570 0.0533            0.50        0.20            0.30

Random Forest | source | pure record minus full
        metric  splits  mean_difference     sd    p2.5  p97.5  share_negative  share

## Comparison Across Models

The table below gathers the paired differences between the releases (anonymised minus source) for Top-1 accuracy and Macro F1, so that the three models can be read side by side. The pure-record feature set uses predictors that are identical in the two releases, so its difference must be exactly zero in every split for a model whose fits are deterministic. The second table reports the largest absolute difference found for that feature set as a consistency check.

In [11]:
side_by_side = release_differences[release_differences.metric.isin(["Top-1 accuracy", "Macro F1"])][
    ["model", "feature_set", "metric", "splits", "mean_difference", "p2.5", "p97.5", "share_negative", "share_zero", "share_positive"]
]

print("REGIONS MINUS SOURCE, ALL MODELS")
print("-" * 110)
print(side_by_side.round(4).to_string(index=False))

print()
print("CONSISTENCY OF THE PURE-RECORD FEATURE SET BETWEEN THE RELEASES")
print("-" * 60)

for model_name in MODELS:
    source_part = select(model_name, "source", "pure_record").set_index("seed")
    regions_part = select(model_name, "regions", "pure_record").set_index("seed")
    largest = max(
        (source_part[column] - regions_part[column]).abs().max() for column in METRICS
    )
    print(f"{MODEL_LABELS[model_name]:<20} largest absolute difference over all splits and metrics: {largest:.2e}")

REGIONS MINUS SOURCE, ALL MODELS
--------------------------------------------------------------------------------------------------------------
              model feature_set         metric  splits  mean_difference    p2.5  p97.5  share_negative  share_zero  share_positive
      Random Forest        full Top-1 accuracy      50           0.0006 -0.0563 0.0543            0.40        0.12            0.48
      Random Forest        full       Macro F1      50           0.0044 -0.0535 0.0794            0.44        0.00            0.56
      Random Forest       no_ai Top-1 accuracy      50           0.0093 -0.0357 0.0552            0.30        0.16            0.54
      Random Forest       no_ai       Macro F1      50           0.0116 -0.0543 0.1092            0.46        0.00            0.54
      Random Forest pure_record Top-1 accuracy      50           0.0000  0.0000 0.0000            0.00        1.00            0.00
      Random Forest pure_record       Macro F1      50           0.000

## Results And Interpretation

The run completed for all 50 splits of every model. The six saved Top-1 Random Forest models use exactly the predictors and settings defined in this notebook, and the seed-42 Top-1 results of the six Random Forest cells are reproduced (differences at most 5.6e-17), so the splits, preprocessing and Random Forest are the same as in the modelling notebooks and in the earlier audit. The test partitions have 52 to 83 rows (mean 66.9) for all three models. All values below are descriptive and carry no significance statement, because the splits overlap and the spread therefore does not represent the sampling variability of a new set of customers. The Logistic Regression and CatBoost models are fitted with fixed settings and without any search, so their scores describe these settings and are not the best that each model family could reach on this table.

### Level and spread of each model

Mean and (standard deviation) over the 50 splits. Top-2 and Top-3 are taken from the same fitted model as Top-1 for every model, so the Random Forest rows are those of the default forest of the earlier audit.

| Model | Cell | Top-1 accuracy | Macro F1 | Top-2 | Top-3 |
|---|---|---|---|---|---|
| Random Forest | source, full | 0.4590 (0.0470) | 0.3506 (0.0543) | 0.7150 (0.0499) | 0.8468 (0.0360) |
| Random Forest | source, no AI | 0.4752 (0.0440) | 0.3780 (0.0582) | 0.7301 (0.0492) | 0.8514 (0.0404) |
| Random Forest | source, pure record | 0.4038 (0.0462) | 0.3252 (0.0518) | 0.6766 (0.0462) | 0.8181 (0.0372) |
| Random Forest | regions, full | 0.4596 (0.0533) | 0.3549 (0.0674) | 0.7107 (0.0383) | 0.8415 (0.0395) |
| Random Forest | regions, no AI | 0.4845 (0.0458) | 0.3896 (0.0607) | 0.7321 (0.0452) | 0.8658 (0.0327) |
| Random Forest | regions, pure record | 0.4038 (0.0462) | 0.3252 (0.0518) | 0.6766 (0.0462) | 0.8181 (0.0372) |
| Logistic Regression | source, full | 0.3137 (0.0526) | 0.2622 (0.0545) | 0.5722 (0.0466) | 0.7466 (0.0481) |
| Logistic Regression | source, no AI | 0.3122 (0.0548) | 0.2638 (0.0578) | 0.5588 (0.0515) | 0.7332 (0.0429) |
| Logistic Regression | source, pure record | 0.4620 (0.0542) | 0.3585 (0.0637) | 0.7412 (0.0467) | 0.8740 (0.0374) |
| Logistic Regression | regions, full | 0.3174 (0.0531) | 0.2639 (0.0497) | 0.5719 (0.0445) | 0.7527 (0.0465) |
| Logistic Regression | regions, no AI | 0.3201 (0.0523) | 0.2722 (0.0591) | 0.5622 (0.0531) | 0.7332 (0.0461) |
| Logistic Regression | regions, pure record | 0.4620 (0.0542) | 0.3585 (0.0637) | 0.7412 (0.0467) | 0.8740 (0.0374) |
| CatBoost | source, full | 0.4585 (0.0505) | 0.3704 (0.0663) | 0.7769 (0.0420) | 0.8936 (0.0335) |
| CatBoost | source, no AI | 0.4806 (0.0508) | 0.3908 (0.0617) | 0.7690 (0.0463) | 0.9030 (0.0314) |
| CatBoost | source, pure record | 0.4018 (0.0494) | 0.3310 (0.0593) | 0.6898 (0.0461) | 0.8540 (0.0329) |
| CatBoost | regions, full | 0.4641 (0.0530) | 0.3793 (0.0697) | 0.7585 (0.0393) | 0.8953 (0.0266) |
| CatBoost | regions, no AI | 0.4838 (0.0454) | 0.4015 (0.0626) | 0.7711 (0.0417) | 0.9012 (0.0342) |
| CatBoost | regions, pure record | 0.4018 (0.0494) | 0.3310 (0.0593) | 0.6898 (0.0461) | 0.8540 (0.0329) |
| Class-prior baseline | both releases | 0.1977 (0.0341) | 0.0487 (0.0068) | 0.3945 (0.0436) | 0.5711 (0.0489) |

- The split-to-split standard deviation is 4.4 to 5.5 points for Top-1 accuracy, 5.0 to 7.0 points for Macro F1, 3.8 to 5.3 points for Top-2 and 2.7 to 4.8 points for Top-3, in every model and cell, so differences of a few points between two rows of the table are of the size of the spread of a single row.
- Every model in every cell is above the class-prior baseline. The lowest Top-1 accuracy is 0.312 (Logistic Regression, source, no AI) against 0.198 for the baseline, and the lowest Top-3 accuracy is 0.733 against 0.571.
- The Random Forest and CatBoost give similar Top-1 accuracy in every cell (0.459 to 0.485 for the full and no-AI feature sets and about 0.40 for the pure-record features). CatBoost has a higher Top-2 and Top-3 accuracy than the Random Forest with the full and no-AI feature sets (by 4 to 6 points), which is a comparison of two sets of rows and not a paired test.
- Logistic Regression behaves differently from the two tree ensembles. With the 55 or 42 predictors its Top-1 accuracy is only 0.312 to 0.320, about 15 points below its own value of 0.462 with the 12 pure-record predictors, and with the pure-record predictors it is the highest of the three models (Top-1 0.462 against 0.404 and 0.402, Top-3 0.874 against 0.818 and 0.854). This pattern is what one would expect from a regularised linear model that receives many one-hot and scaled columns for 431 rows, but the notebook does not test this explanation.
- The ranking of the models therefore depends on the feature set, and the table gives no support for a statement that one model family is better than the others in general. The Random Forest was retained in the modelling notebooks and was not selected on a comparison of this kind.

### Source release against anonymised release

Anonymised minus source, paired within each split (mean difference, with the 2.5th to 97.5th percentile of the 50 differences in brackets):

| Model | Feature set | Top-1 accuracy | Macro F1 | Top-2 | Top-3 |
|---|---|---|---|---|---|
| Random Forest | full | +0.0006 (-0.0563 to +0.0543) | +0.0044 (-0.0535 to +0.0794) | -0.0043 (-0.0689 to +0.0649) | -0.0053 (-0.0555 to +0.0429) |
| Random Forest | no AI | +0.0093 (-0.0357 to +0.0552) | +0.0116 (-0.0543 to +0.1092) | +0.0020 (-0.0466 to +0.0439) | +0.0144 (-0.0285 to +0.0746) |
| Random Forest | pure record | 0.0000 | 0.0000 | 0.0000 | 0.0000 |
| Logistic Regression | full | +0.0037 (-0.0976 to +0.0968) | +0.0018 (-0.0869 to +0.0855) | -0.0004 (-0.0868 to +0.0593) | +0.0060 (-0.0588 to +0.0941) |
| Logistic Regression | no AI | +0.0078 (-0.0712 to +0.0966) | +0.0083 (-0.0777 to +0.0971) | +0.0034 (-0.0657 to +0.1161) | 0.0000 (-0.0789 to +0.0723) |
| Logistic Regression | pure record | 0.0000 | 0.0000 | 0.0000 | 0.0000 |
| CatBoost | full | +0.0056 (-0.0445 to +0.0645) | +0.0089 (-0.0453 to +0.0959) | -0.0184 (-0.0822 to +0.0408) | +0.0017 (-0.0444 to +0.0451) |
| CatBoost | no AI | +0.0032 (-0.0428 to +0.0663) | +0.0107 (-0.0533 to +0.1078) | +0.0021 (-0.0423 to +0.0585) | -0.0017 (-0.0298 to +0.0292) |
| CatBoost | pure record | 0.0000 | 0.0000 | 0.0000 | 0.0000 |

- For the pure-record features, whose predictors are identical in the two releases, the difference is exactly zero in all 50 splits for every metric and every model (largest absolute difference 0.00e+00). This confirms that the splits, seeds and code are the same for both releases, also for Logistic Regression and CatBoost.
- For the full and no-AI feature sets, which contain the generalised predictors, the mean differences of the three models lie between -1.8 and +1.4 points over all four metrics. The mean Top-1 difference is small and positive for all three models (+0.1 to +0.9 points), and the Top-1 difference is negative in 30 to 40 percent of the splits, zero in 12 to 24 percent and positive in 40 to 54 percent.
- Every 2.5th to 97.5th percentile range contains zero, and the ranges (about 9 to 19 points wide for Top-1 accuracy) are of the size of the split-to-split variation of a single cell. The largest mean difference in absolute terms is the Top-2 accuracy of CatBoost with the full feature set (-1.8 points, negative in 56 percent of the splits), and it also lies within its range.
- The three model families, which differ strongly in level, therefore show the same picture as the Random Forest of the earlier audit, namely that the anonymised release gives results within the spread of ordinary split-to-split variation of the source release. This describes what was observed and does not show that anonymisation has no effect, particularly because the table is small (431 states).

### With and without the AI features

Full feature set minus no-AI feature set, paired within each split (with AI minus without AI):

| Model | Release | Top-1 accuracy | Macro F1 | Top-2 | Top-3 |
|---|---|---|---|---|---|
| Random Forest | source | -0.0162 (-0.0794 to +0.0469) | -0.0274 (-0.1613 to +0.0394) | -0.0151 (-0.0987 to +0.0586) | -0.0046 (-0.0570 to +0.0533) |
| Random Forest | regions | -0.0249 (-0.1161 to +0.0349) | -0.0347 (-0.1597 to +0.0461) | -0.0214 (-0.0846 to +0.0445) | -0.0243 (-0.0706 to +0.0311) |
| Logistic Regression | source | +0.0014 (-0.0760 to +0.0991) | -0.0017 (-0.0846 to +0.1144) | +0.0134 (-0.0434 to +0.0786) | +0.0134 (-0.0788 to +0.0816) |
| Logistic Regression | regions | -0.0027 (-0.0716 to +0.0798) | -0.0083 (-0.0890 to +0.0798) | +0.0097 (-0.0442 to +0.0748) | +0.0195 (-0.0634 to +0.0827) |
| CatBoost | source | -0.0221 (-0.0891 to +0.0427) | -0.0204 (-0.1111 to +0.0686) | +0.0079 (-0.0745 to +0.0822) | -0.0094 (-0.0665 to +0.0347) |
| CatBoost | regions | -0.0197 (-0.0879 to +0.0503) | -0.0221 (-0.1306 to +0.1301) | -0.0125 (-0.0748 to +0.0384) | -0.0060 (-0.0557 to +0.0437) |

- For the Random Forest and CatBoost the mean Top-1 accuracy is 1.6 to 2.5 points lower with the AI features than without them in both releases, and the Top-1 difference is negative in 54 to 70 percent of the splits. For Logistic Regression the Top-1 difference is close to zero (+0.1 and -0.3 points), while its Top-2 and Top-3 are 1.0 to 2.0 points higher with the AI features.
- Every percentile range contains zero. The AI features therefore showed no consistent advantage in these data for any of the three models. The result describes the fitted models and is not interpreted as a causal effect of the features, and part of the AI-derived information is time-stamped after the prediction time (see the modelling notebooks), which these comparisons do not address.

### Documented-record features only

Pure-record minus full feature set, paired within each split:

| Model | Release | Top-1 accuracy | Macro F1 | Top-2 | Top-3 |
|---|---|---|---|---|---|
| Random Forest | source | -0.0552 (-0.1770 to +0.0480) | -0.0254 (-0.1490 to +0.0749) | -0.0384 (-0.1226 to +0.0269) | -0.0287 (-0.0894 to +0.0525) |
| Random Forest | regions | -0.0558 (-0.1687 to +0.0386) | -0.0298 (-0.1477 to +0.0633) | -0.0340 (-0.1065 to +0.0290) | -0.0235 (-0.1038 to +0.0560) |
| Logistic Regression | source | +0.1484 (-0.0112 to +0.2805) | +0.0963 (-0.0441 to +0.2360) | +0.1690 (+0.0622 to +0.2673) | +0.1274 (+0.0306 to +0.2157) |
| Logistic Regression | regions | +0.1447 (+0.0234 to +0.2758) | +0.0945 (-0.0363 to +0.2417) | +0.1694 (+0.0594 to +0.2618) | +0.1214 (+0.0348 to +0.2226) |
| CatBoost | source | -0.0567 (-0.1576 to +0.0518) | -0.0394 (-0.1518 to +0.0516) | -0.0871 (-0.1786 to 0.0000) | -0.0396 (-0.1023 to 0.0000) |
| CatBoost | regions | -0.0623 (-0.1889 to +0.0354) | -0.0484 (-0.2147 to +0.0457) | -0.0688 (-0.1289 to -0.0029) | -0.0413 (-0.1043 to +0.0106) |

- For the Random Forest and CatBoost the 12 documented-record predictors give lower scores than all 55 predictors, by 5.5 to 6.2 points in Top-1 accuracy and by 2.4 to 8.7 points in Top-2 and Top-3, and the Top-1 difference is negative in 78 to 82 percent of the splits.
- For Logistic Regression the sign is the opposite and the size is large, with the pure-record features giving +14.5 to +14.8 points of Top-1 accuracy and +12 to +17 points of Top-2 and Top-3 accuracy, positive in 94 to 100 percent of the splits. For its Top-2 and Top-3 accuracy the whole 2.5th to 97.5th percentile range is above zero in both releases.
- With only these 12 predictors all three models are still far above the class-prior baseline, so most of the usable signal is in the documented category counts and values.

### Class-prior baseline

The class-prior baseline ranks the classes by their frequency in the training partition and gives a mean Top-1, Top-2 and Top-3 accuracy of 0.198, 0.395 and 0.571 over the 50 splits. It is identical for both releases and for all models because it uses only the training labels, and the Macro F1 of the baseline is 0.049.

### Limitations

- The table has 431 prediction states and 198 customers. The 50 splits are drawn from the same states, so the spread understates the uncertainty about new customers.
- Logistic Regression and CatBoost are fitted without tuning, with the settings stated above, so the model comparison reflects those settings and the CatBoost scores are not comparable with the CatBoost rows of the modelling notebooks.
- The comparison between the models is made on separate fits and is not a paired analysis, and the validation partition is not used for any selection.
- The comparison measures predictive utility only and does not measure the privacy of the anonymised release.